In [1]:
%load_ext autoreload
%autoreload 2

In [20]:
from pathlib import Path

import numpy as np
import rasterio
import matplotlib.pyplot as plt

import torch
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.strategies import DDPStrategy
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

from dataset.dataset import SegmentationDataset
from dataset.datamodule import SegmentationModule
from utils.albumentation import augment
from utils.plotting import plot_image_mask

In [ ]:
eval_root = Path("C:/Users/serr_da/Documents/spatio_temp_seg_flair1/evaluation")
data_root = Path("C:/Users/serr_da/Documents/Datasets/flair_1_toy_dataset")
subfolder = Path()

In [ ]:
model = SegmentationModule(
    data_dir=data_root,
    num_classes=19,
    in_channels=5,
    ignore_index=0,
    augment=augment,
    batch_size=16,
    num_workers=4,
)

In [5]:
model.setup(stage="fit")
train_dataloader = model.train_dataloader()
val_dataloader = model.val_dataloader()

model.setup(stage="test")
test_dataloader = model.test_dataloader()

Lenght full dataset: 200
Lenght training dataset: 160
Lenght validation dataset: 40
Lenght test dataset: 50


In [19]:
# plot_image_mask(data_loader=train_dataloader, take=10, normalized=True)

In [ ]:
class_weights = torch.tensor([
    0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 
])

In [ ]:
# Checkpoints
checkpoint_root = Path("")
checkpoint_dir = checkpoint_root / subfolder / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True, parents=True)

experiment = ""
experiment_dir = root / experiment / "checkpoints"
checkpoint_path = experiment_dir / ""

model = SegmentationModule.load_from_checkpoint(
    checkpoint_path=checkpoint_path,
    weights_only=False,
    data_dir=root,
    num_classes=19,
    in_channels=5,
    ignore_index=19,
    class_weights=class_weights,
    learning_rate=1e-03,
    batch_size=16,
    num_workers=4
)

In [ ]:
# Callbacks
logger_root = Path("")
logger_dir = logger_root / subfolder
logger_dir.mkdir(parents=True, exist_ok=True)

csv_logger = CSVLogger(
    save_dir=logger_dir,
    name="csv_logs"
)

tb_logger = TensorBoardLogger(
    logger_dir, 
    name="tb_logs"
)

checkpoint = ModelCheckpoint(
    monitor="val_loss",
    dirpath=checkpoint_dir,
    filename="ckpt-{epoch:02d}-{val_loss:.3f}-{val_iou:.3f}",
    save_top_k=2,
    mode="min",
    save_weights_only=True
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    min_delta=0.00,
    patience=20,
    mode="max",
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

callbacks = [
    checkpoint,
    early_stopping,
    lr_monitor
]


# Define trainer
trainer = Trainer(
    accelerator="gpu",
    max_epochs=200,
    # devices=[1],
    devices="auto",
    # strategy=DDPStrategy(find_unused_parameters=True),
    callbacks=callbacks,
    enable_progress_bar=True,
    logger=[csv_logger, tb_logger],
    log_every_n_steps=10
)

# Train model
trainer.fit(model)